# **NetMHCpan wrapper development notebook**

### **Goals**:

Given HLA/Epitope table predict affinity values using NetMHCpan

Make smart match: A02 = A0201, A0202 etc

Given epitope table only predict affinity for best HLA

Don't mess affinity types: keep affinity rank and affinity value separate

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import subprocess
import itertools
import resource
import xlrd

### **Config**

In [5]:
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))


PATH_TO_NETMHCPAN = Path('/home/aalarkin/Work/netmhcpan_wrapper/netMHCpan-4.2')
TMPDIR = Path('/home/aalarkin/Work/netmhcpan_wrapper/tmp')
PATH_TO_SUPERGROUPS = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/mhc_supergroups.txt')


ALPHABET = {'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y'}


### **Utilities**

1. Creating and validating peplists

In [6]:
def validate_peptide(seq):
    if not len(seq) > 0 or type(seq) != str:
        return(False)

    for a in list(seq):
        if a not in ALPHABET:
            return(False)
        
    return(True)


def generate_peplist(pep_array, output_file):

    with open(output_file, mode = 'w') as o:
        for s in pep_array:
            if validate_peptide(s) == True:
                o.write(f'{s}\n')
            else:
                raise ValueError(f'Peptide {s} is invalid')

    

In [4]:
peptides = ['WWWWWWTTTYYYYY', 'GLGLGIIIWWVV', 'WWWYYYKKKQRS']
generate_peplist(peptides, f'{TMPDIR}/pep_test.txt')
subprocess.run(f'cat {TMPDIR}/pep_test.txt', shell = True)

WWWWWWTTTYYYYY
GLGLGIIIWWVV
WWWYYYKKKQRS


CompletedProcess(args='cat /home/aalarkin/Work/netmhcpan_wrapper/tmp/pep_test.txt', returncode=0)

2. Creating and validating list of hla alleles with smartmatch

for smartmatch a list of hla supergroups was created. NetMHCpan allelelist.txt was used

example:

HLA-A02;HLA-A\*02:01, HLA-A\*02:01 ... \n

Total: 528 supergroups covering 11,180 alleles

In [7]:
def match_hla(hla_array):
    outputs = []
    sg = {}


    with open(PATH_TO_SUPERGROUPS, mode = 'r') as f:
        for i in f:
            line = i.rstrip('\n').split(';')
            sg[line[0]] = set(line[1].split(','))

    known_alleles = set(itertools.chain(*list(sg.values())))

    for i in hla_array:
        if i in sg:
            outputs.append([','.join(sg[i])])
        elif i in known_alleles:
            outputs.append([i])
        else:
            outputs.append([np.nan])

    return(outputs)




In [6]:
hlas = ['HLA-A*02:01', 'HLA-A02', 'ababa']
match_hla(hlas)

[['HLA-A*02:01'],
 ['HLA-A*02:230,HLA-A*02:206,HLA-A*02:127,HLA-A*02:454,HLA-A*02:538,HLA-A*02:39,HLA-A*02:607,HLA-A*02:255,HLA-A*02:262,HLA-A*02:303,HLA-A*02:589,HLA-A*02:535,HLA-A*02:528,HLA-A*02:341,HLA-A*02:61,HLA-A*02:280,HLA-A*02:681,HLA-A*02:560,HLA-A*02:682,HLA-A*02:215,HLA-A*02:251,HLA-A*02:343,HLA-A*02:362,HLA-A*02:508,HLA-A*02:650,HLA-A*02:103,HLA-A*02:523,HLA-A*02:186,HLA-A*02:197,HLA-A*02:139,HLA-A*02:732,HLA-A*02:808,HLA-A*02:237,HLA-A*02:374,HLA-A*02:154,HLA-A*02:561,HLA-A*02:171,HLA-A*02:89,HLA-A*02:424,HLA-A*02:428,HLA-A*02:74,HLA-A*02:08,HLA-A*02:753,HLA-A*02:66,HLA-A*02:780,HLA-A*02:376,HLA-A*02:45,HLA-A*02:324,HLA-A*02:653,HLA-A*02:47,HLA-A*02:581,HLA-A*02:294,HLA-A*02:232,HLA-A*02:36,HLA-A*02:104,HLA-A*02:381,HLA-A*02:01,HLA-A*02:562,HLA-A*02:367,HLA-A*02:664,HLA-A*02:160,HLA-A*02:750,HLA-A*02:550,HLA-A*02:150,HLA-A*02:310,HLA-A*02:281,HLA-A*02:106,HLA-A*02:617,HLA-A*02:10,HLA-A*02:191,HLA-A*02:14,HLA-A*02:267,HLA-A*02:351,HLA-A*02:820,HLA-A*02:711,HLA-A*02:765,HLA

### **Core functions**

1. Given epitope table only predict affinity for best HLA

In [ ]:
def pan_predict(epitopes, workdir):

    generate_peplist(epitopes, f'{workdir}/peplist.txt')

    sg = {}
    with open(PATH_TO_SUPERGROUPS, mode = 'r') as f:
        for i in f:
            line = i.rstrip('\n').split(';')
            sg[line[0]] = set(line[1].split(','))


    known_alleles = list(itertools.chain(*list(sg.values())))
    for i in range(len(known_alleles)):
        if '*' in known_alleles[i]:
            known_alleles[i] = ''.join(known_alleles[i].split('*'))

   # known_alleles = ','.join(known_alleles)
    
    batch_size = 20

    for i in range(0, len(known_alleles), batch_size):
        batch = known_alleles[i:i+batch_size]
        alleles_str = ','.join(batch)
        
        batch_output = f'{workdir}/results_batch_{i//batch_size}.xls'
        
        cmd = f'{PATH_TO_NETMHCPAN}/netMHCpan -a {alleles_str} -p {workdir}/peplist.txt -xls -xlsfile {batch_output} -BA'
        
        subprocess.run(cmd, shell=True, check=True)
    

In [76]:
epitopes = ['WWWWWWWWW', 'YYYYYYYYY', 'GGGGGGGGGGGGGG']
pan_predict(epitopes, TMPDIR)

TypeError: pan_predict() missing 1 required positional argument: 'species'

2. Given HLA/Epitope table predict affinity values using NetMHCpan

In [ ]:
def predict_affinnity(epitope, hla):
    if not validate_peptide(epitope):
        raise ValueError(f'Epitope validation for {epitope} failed')
    

    hla_string = None

    sg = {}
    with open(PATH_TO_SUPERGROUPS, mode = 'r') as f:
        for i in f:
            line = i.rstrip('\n').split(';')
            sg[line[0]] = set(line[1].split(','))

    known_alleles = set(itertools.chain(*list(sg.values())))

    if hla in sg.keys():
        hla_string = list(sg[hla])

        for i in range(len(hla_string)):
            hla_string[i] = ''.join(hla_string[i].split('*'))
        
        hla_string = ','.join(hla_string)

    elif hla in known_alleles:
        hla_string = ''.join(hla.split('*'))

    else:
        raise ValueError(f'Unknown MHC allele or supergroup: {hla}')
    
    print(epitope, hla_string)

    subprocess.run(f'echo "{epitope}" > {TMPDIR}/temp_epitope.txt', shell=True, check=True)
    subprocess.run(f'{PATH_TO_NETMHCPAN}/netMHCpan -a {hla_string} -p {TMPDIR}/temp_epitope.txt -xls -xlsfile {TMPDIR}/affinity_pred.xls -BA -mode 0' , shell=True, check=True)
    subprocess.run(f'rm {TMPDIR}/temp_epitope.txt', shell=True, check=True)

   # df = pd.read_excel(f'{TMPDIR}/affinity_pred.xls', engine = 'xlrd')
   # display(df)
    

    
   

    

    

    

In [75]:
predict_affinnity('AWWGGGGGG', 'HLA-A02')

AWWGGGGGG HLA-A02:160,HLA-A02:633,HLA-A02:187,HLA-A02:749,HLA-A02:299,HLA-A02:132,HLA-A02:823,HLA-A02:120,HLA-A02:171,HLA-A02:466,HLA-A02:415,HLA-A02:391,HLA-A02:770,HLA-A02:664,HLA-A02:45,HLA-A02:137,HLA-A02:551,HLA-A02:80,HLA-A02:144,HLA-A02:179,HLA-A02:292,HLA-A02:168,HLA-A02:427,HLA-A02:49,HLA-A02:510,HLA-A02:78,HLA-A02:209,HLA-A02:290,HLA-A02:557,HLA-A02:721,HLA-A02:139,HLA-A02:795,HLA-A02:684,HLA-A02:616,HLA-A02:682,HLA-A02:242,HLA-A02:386,HLA-A02:653,HLA-A02:03,HLA-A02:400,HLA-A02:471,HLA-A02:625,HLA-A02:155,HLA-A02:374,HLA-A02:679,HLA-A02:666,HLA-A02:91,HLA-A02:707,HLA-A02:431,HLA-A02:713,HLA-A02:542,HLA-A02:392,HLA-A02:402,HLA-A02:488,HLA-A02:531,HLA-A02:183,HLA-A02:619,HLA-A02:180,HLA-A02:351,HLA-A02:464,HLA-A02:541,HLA-A02:71,HLA-A02:821,HLA-A02:202,HLA-A02:154,HLA-A02:246,HLA-A02:368,HLA-A02:742,HLA-A02:543,HLA-A02:362,HLA-A02:39,HLA-A02:145,HLA-A02:662,HLA-A02:249,HLA-A02:333,HLA-A02:348,HLA-A02:409,HLA-A02:718,HLA-A02:598,HLA-A02:539,HLA-A02:790,HLA-A02:564,HLA-A02:119,HL